# Project 3: Production Data Quality Gate and Automated Audit Pipeline

**Module**: 13_end_to_end_projects  
**Notebook**: 03_data_quality_pipeline.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: Building an automated data quality pipeline: schema conformance, null auditing, uniqueness verification, range checks, foreign key integrity, and generating an automated data health scorecard.
- **Why does this topic exist?**: Garbage in, garbage out. Production data platforms must halt bad data at ingestion gates.
- **What problem does it solve?**: Handling corrupt records in /opt/spark-data/quality/orders_dirty.csv without crashing the job.
- **Where is it used in real-world Data Engineering?**: Implementing Great Expectations / Deequ style data quality gates.

---

<!-- curriculum-enrichment:v1 -->

## Learning Contract

| Attribute | Value |
|---|---|
| **Priority** | Important |
| **Estimated effort** | 4–6 h |
| **Prerequisites** | Data validation, SQL, pipeline design |
| **Required depth** | Working knowledge: can build automated quality gates and audit outputs |
| **Data Engineer** | Implement quality checks that block bad data and produce actionable evidence. |
| **Senior Data Engineer** | Design quality dimensions, thresholds, and quarantine workflows. |
| **Spark specialist** | Reason about completeness, consistency, timeliness, and anomaly detection trade-offs. |
| **Next topic** | `13_end_to_end_projects/04_streaming_analytics_pipeline.ipynb` |

> **How to use this notebook**: read the problem first, then run the code, then connect the
> mechanism to the real-world cases and production notes. The goal is not to memorize API
> syntax; it is to be able to explain and defend a design decision.

---

## Problem and Motivation

### Why this concept exists

A data-quality pipeline turns implicit assumptions into explicit checks: keys are unique, required fields are
present, referential integrity holds, and row counts are within expected bounds. The goal is not to reject every
imperfection; it is to stop bad data from reaching consumers while preserving evidence and allowing legitimate
variation.

### What happens if you ignore it

A team can still write code that runs on small data. The failure appears later: a job that
looks correct becomes slow, unstable, or expensive at scale, and the team lacks the vocabulary
to diagnose which Spark mechanism is responsible. This notebook builds that vocabulary and
ties it to observable behavior in the lab.

### A small mental model

Think of the topic as a contract between the logical work you describe and the physical work
Spark performs. The contract has a cost, a failure mode, and a set of trade-offs. The sections
below make each of those explicit.

---

## Core Concepts

### Definitions

| Term | Plain-language meaning | Technical meaning |
|---|---|---|
| **Completeness** | Required data present | Null checks and missing-partition checks |
| **Uniqueness** | No duplicate keys | Duplicate counts by business key |
| **Referential integrity** | Foreign keys resolve | Anti-joins against dimensions |
| **Range/domain** | Values are plausible | Min/max, accepted sets, type checks |
| **Timeliness** | Data arrives on time | Freshness and partition-arrival checks |
| **Quarantine** | Isolate bad records | Preserve evidence without blocking good data |

### How it works, step by step

1. Read the incoming batch and compute quality metrics.
2. Compare metrics with thresholds and historical baselines.
3. Classify the batch as pass, warn, or fail.
4. Quarantine failing records or partitions with metadata.
5. Publish only passing data to the curated layer.
6. Record an audit report with counts and reasons.

### What to keep in mind

- The mechanism is not free; it consumes CPU, memory, network, or disk.
- The right choice depends on data size, distribution, latency target, and failure tolerance.
- A plan or UI observation is stronger evidence than an intuition about what "should" happen.

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Data_Quality_Project")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

## 3. Reading Dirty Data and Defining Quality Constraints

In [ ]:
dirty_df = spark.read.option("header", True).option("inferSchema", True).csv("/opt/spark-data/quality/orders_dirty.csv")
print(f"Total input rows in audit pool: {dirty_df.count():,}")
dirty_df.show(3)

## 4. Rule-Based Record Categorization (Valid vs Quarantined)

In [ ]:
# Define quality checks
# Rule 1: order_id is not null and positive
# Rule 2: customer_id is not null
# Rule 3: quantity > 0 and unit_price > 0
# Rule 4: discount between 0.0 and 1.0

flagged_df = (
    dirty_df
    .withColumn("is_null_id", F.col("order_id").isNull() | F.col("customer_id").isNull())
    .withColumn("is_invalid_qty", F.col("quantity").isNull() | (F.col("quantity") <= 0))
    .withColumn("is_invalid_price", F.col("unit_price").isNull() | (F.col("unit_price") <= 0))
    .withColumn("is_invalid_discount", F.col("discount").isNull() | (F.col("discount") < 0.0) | (F.col("discount") > 1.0))
    .withColumn("is_valid", ~(F.col("is_null_id") | F.col("is_invalid_qty") | F.col("is_invalid_price") | F.col("is_invalid_discount")))
)

valid_records = flagged_df.filter("is_valid = True").drop("is_null_id", "is_invalid_qty", "is_invalid_price", "is_invalid_discount", "is_valid")
quarantine_records = flagged_df.filter("is_valid = False")

print(f"Valid Clean Records:     {valid_records.count():,}")
print(f"Quarantined Bad Records: {quarantine_records.count():,}")

---

## Experiment: Generating Automated Data Quality Scorecard

- **1. What are we testing?**: Aggregate metrics across all quality rules into an executive summary report.
- **2. What do we expect?**: A structured DataFrame reporting error counts and pass percentages per rule.

In [ ]:
total_rows = dirty_df.count()

scorecard = flagged_df.select(
    F.lit(total_rows).alias("total_records"),
    F.sum(F.col("is_null_id").cast("int")).alias("failed_null_ids"),
    F.sum(F.col("is_invalid_qty").cast("int")).alias("failed_quantity_check"),
    F.sum(F.col("is_invalid_price").cast("int")).alias("failed_price_check"),
    F.sum(F.col("is_invalid_discount").cast("int")).alias("failed_discount_check"),
    F.sum(F.col("is_valid").cast("int")).alias("passed_records")
)

print("=== DATA QUALITY SCORECARD ===")
scorecard.show(vertical=True)

**3. What happened?**: Generated complete data quality audit report.  
**4. Why did it happen?**: Aggregated rule flags into summary counters.  
**5. Practical Engineering Lesson**: Always publish data quality scorecards to monitoring systems (Datadog/Prometheus) to alert on ingestion anomalies.

---

## Real-World Use Cases

> The scenarios below are illustrative. Company names are omitted, and any numbers are hypothetical unless they are measured in this lab.

### Use Case 1: Telecom CDR validation

- **Industry**: Telecom
- **Scenario**: Call detail records arrive from many switches with occasional duplicates and missing fields.
- **Problem**: Bad records can distort billing and network analytics.
- **Why this concept is relevant**: A quality gate checks key uniqueness, required fields, and call-duration ranges before publishing.
- **Implementation idea**: Quarantine invalid records, publish valid ones, and alert on the failure rate.
- **Expected result**: Billing-grade data with an auditable exception path.
- **Alternative**: Reject the entire batch on any error.
- **Trade-offs**: Rejecting everything is safe but can halt the business; quarantine balances safety and progress.

### Use Case 2: Healthcare claims audit

- **Industry**: Healthcare
- **Scenario**: Claims data must satisfy referential integrity with member and provider dimensions.
- **Problem**: Orphan claims break downstream reporting and compliance.
- **Why this concept is relevant**: Anti-join checks identify orphan records before publication.
- **Implementation idea**: Quarantine orphan claims and report them to the data owner.
- **Expected result**: A clean claims mart with a clear exception list.
- **Alternative**: Drop orphan claims silently.
- **Trade-offs**: Silent drops hide revenue leakage and compliance risk.

### Use Case 3: E-commerce catalog quality

- **Industry**: Retail
- **Scenario**: Product attributes are incomplete or inconsistent across suppliers.
- **Problem**: Search and recommendations degrade when key attributes are missing.
- **Why this concept is relevant**: Completeness and domain checks score each product and route low-quality records for enrichment.
- **Implementation idea**: Publish products that meet the threshold and flag the rest for review.
- **Expected result**: Higher catalog quality with a measurable backlog.
- **Alternative**: Block all incomplete products.
- **Trade-offs**: Blocking everything reduces selection; scoring allows partial publication with visibility.

---

## Business Value

- Fewer downstream incidents caused by bad data.
- Auditable evidence for compliance.
- Clear ownership of data defects.
- Higher trust in dashboards and models.

> **Note on numbers**: this repository does not claim production benchmarks. If you want to
> quantify a business case, measure the job in your own environment (runtime, shuffle bytes,
> spill, infrastructure cost) before and after the change.

---

## Production Engineering

### What changes when this runs in production

- Define thresholds with data owners, not in isolation.
- Quarantine rather than silently dropping records.
- Track quality metrics over time to detect drift.
- Make the gate idempotent and rerunnable.
- Alert on failure rates, not only absolute counts.

### Investigation checklist

- [ ] Open the Spark UI and identify the relevant tab for this topic.
- [ ] Compare the metric before and after a configuration or code change.
- [ ] Check driver and executor logs for warnings or retries.
- [ ] Confirm that the fix is measured, not assumed.

### Common production mistakes

1. Copying a local-lab setting into a production cluster without sizing it.
2. Optimizing a single job without checking the downstream SLA and data-quality impact.
3. Treating a successful run as proof of correctness; verify row counts, keys, and invariants.
4. Forgetting that retries and backfills must be idempotent.

### Experiment: Add a quality gate

**Steps**

1. Create a batch with duplicate keys and null required fields.
2. Run completeness, uniqueness, and referential-integrity checks.
3. Verify that invalid records are quarantined and valid records are published.

**What to observe**: The gate separates valid and invalid data and records the reason.

**Healthy result**: Published data passes all checks and the quarantine is auditable.

**Problematic result**: The gate drops bad records without evidence.

**Engineering takeaway**: Quality gates must preserve evidence, not just block data.

---

## Trade-offs and Alternatives

### Advantages

- Prevents bad data propagation
- Auditable
- Actionable exceptions

### Limitations

- Thresholds require tuning
- Adds latency
- Can block legitimate edge cases

### When to use it

- Billing data
- Regulated data
- Catalog quality

### When not to use it

- Silent drops
- All-or-nothing gates
- Static thresholds without review

### Alternatives inside Spark

- Validation queries
- Anti-joins
- Quarantine tables

### Alternatives outside Spark

- Data quality platforms
- Great Expectations
- Contract testing

### Performance implications

Every choice has a cost. Measure runtime, shuffle read/write, spill, GC time, and task
duration percentiles before and after the change. A local four-core cluster is useful for
observing mechanics, but it is not a production benchmark.

---

## Practical Exercises

### Exercise 1 (Beginner): Write Quarantine Table
**Problem Statement**: Write quarantined records to /opt/spark-data/warehouse/quarantine_orders.  
**Input / Context**: quarantine_records

In [ ]:
# Write your solution for Exercise 1 here

### Exercise 2 (Intermediate): Referential Integrity Check
**Problem Statement**: Check if customer_ids in dirty_df exist in customers.csv.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 2 here

### Exercise 3 (Advanced): Automated Quality Threshold Gate
**Problem Statement**: Raise an exception if pass rate is below 90%.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 3 here

---

## Production Debugging Exercise

### The gate that blocked everything

**Scenario**: A new quality rule rejects 100% of a batch after a legitimate upstream change.

**Your task**: identify the most likely root cause, propose one diagnostic step, and propose
one fix. Write your answer before opening the model approach.

**Hints**

1. Compare the rule with the upstream schema change.
2. Check whether the threshold or domain assumption is still valid.
3. Use a warn mode or quarantine while the rule is reviewed.

<details>
<summary>Model approach (open after trying)</summary>

1. Reproduce the issue with the smallest possible dataset.
2. Use the Spark UI and the physical plan to locate the stage or operator that dominates.
3. Change one variable at a time and re-measure.
4. Record the before/after metric so the fix is defensible.

The exact fix depends on the evidence. The important habit is to separate allocation,
scheduling, shuffle, memory, and data-quality causes instead of guessing.

</details>

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Write Quarantine Table):
```python
quarantine_records.write.mode('overwrite').parquet('/opt/spark-data/warehouse/quarantine_orders')
```

#### Solution 2 (Referential Integrity Check):
```python
custs = spark.read.option('header', True).csv('/opt/spark-data/retail/customers.csv')
orphan_orders = dirty_df.join(custs,'customer_id','left_anti')
print('Orphan orders count:', orphan_orders.count())
```

#### Solution 3 (Automated Quality Threshold Gate):
```python
pass_rate = valid_records.count() / total_rows
assert pass_rate >= 0.90, f'Quality threshold failed! Pass rate: {pass_rate:.2%}'
print('Quality gate passed!')
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Quality Gates**: Quarantine bad rows without crashing the pipeline.
- **Scorecarding**: Quantify data quality metrics across batch runs.

### Essential Technical Terms
- `Quarantine`: Isolating corrupt records into a separate dead-letter queue table.
- `Data Quality Gate`: Automated threshold validation step.

### Common Engineering Mistakes
- **Silently dropping bad data without audit**: Hides upstream data ingestion bugs.

### Interview Questions on This Topic
- **Q: How do you handle bad records without failing an entire batch job?**
  - *A*: Flag records using boolean rule expressions, split the DataFrame into valid and quarantine subsets, write valid rows to the lake, and write quarantine rows to a dead-letter queue with audit metadata.


### More Interview Questions

- **Q: What are the main data-quality dimensions?**
  - *A*: Completeness, uniqueness, referential integrity, validity/range, timeliness, and consistency are common dimensions.
- **Q: Why quarantine instead of dropping bad records?**
  - *A*: Quarantine preserves evidence for investigation and reprocessing while allowing valid records to proceed.

### What to Study Next
Open 04_streaming_analytics_pipeline.ipynb for real-time event analytics.

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')